# Human Development Index Analysis
This notebook mirrors the reproducible analysis and model training implemented in `train_model.py`.

In [ ]:
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = BASE_DIR / 'Dataset' / 'HumanDevelopmentIndex.csv'
FEATURES = ['Life Expectancy', 'Mean Years of Schooling', 'Expected Years of Schooling', 'Gross National Income per Capita']
TARGET = 'HDI'


In [ ]:
data = pd.read_csv(DATA_PATH)
display(data.head())
data.info()
display(data.describe())
print('Shape:', data.shape)
print('Countries:', data['Country'].unique())
display(data.isna().sum().rename('Missing values'))
numeric_columns = data.select_dtypes(include='number').columns
data[numeric_columns] = data[numeric_columns].fillna(data[numeric_columns].mean())


In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.stripplot(data=data, x='Development Group', y=TARGET, jitter=.18, ax=axes[0, 0])
axes[0, 0].set_title('HDI Distribution by Development Group')
sns.regplot(data=data, x='Life Expectancy', y=TARGET, ax=axes[0, 1])
axes[0, 1].set_title('Life Expectancy vs HDI')
sns.regplot(data=data, x='Mean Years of Schooling', y=TARGET, ax=axes[1, 0])
axes[1, 0].set_title('Mean Years of Schooling vs HDI')
sns.heatmap(data[FEATURES + [TARGET]].corr(), annot=True, cmap='YlGnBu', fmt='.2f', ax=axes[1, 1])
axes[1, 1].set_title('Correlation Heatmap')
plt.tight_layout()


In [ ]:
X, y = data[FEATURES], data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=42)
model = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('regressor', LinearRegression())])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(f'R² score: {r2_score(y_test, predictions):.4f}')
display(pd.DataFrame({'Actual HDI': y_test.to_numpy(), 'Predicted HDI': predictions}).round(4))
with open(BASE_DIR / 'hdi_model.pkl', 'wb') as model_file:
    pickle.dump(model, model_file)
